# Contrarian Exit Strategy

**The Insight:** We enter on fear (SOPR < 1), so why not exit on greed (SOPR > threshold)?

**Buy Fear, Sell Greed:**
- Entry: SOPR < 1 AND STH SOPR < 1 (capitulation)
- Exit: SOPR > X OR STH SOPR > Y (euphoria)

This is the full contrarian approach - no arbitrary price-based stops.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Buy Fear, Sell Greed! 🐻➡️🐂")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")

df = sopr.join(sopr_sth, how='inner').join(price, how='inner').sort_index()
df = df[df.index >= '2018-12-15']

close = df['price']
print(f"Data: {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")

In [ ]:
# Look at SOPR distribution to understand exit thresholds
print("SOPR Statistics:")
print(f"  Min: {df['sopr'].min():.3f}")
print(f"  Max: {df['sopr'].max():.3f}")
print(f"  Median: {df['sopr'].median():.3f}")
print(f"  Mean: {df['sopr'].mean():.3f}")
print(f"  75th percentile: {df['sopr'].quantile(0.75):.3f}")
print(f"  90th percentile: {df['sopr'].quantile(0.90):.3f}")
print(f"  95th percentile: {df['sopr'].quantile(0.95):.3f}")

print(f"\nSTH SOPR Statistics:")
print(f"  Min: {df['sopr_sth'].min():.3f}")
print(f"  Max: {df['sopr_sth'].max():.3f}")
print(f"  Median: {df['sopr_sth'].median():.3f}")
print(f"  75th percentile: {df['sopr_sth'].quantile(0.75):.3f}")
print(f"  90th percentile: {df['sopr_sth'].quantile(0.90):.3f}")
print(f"  95th percentile: {df['sopr_sth'].quantile(0.95):.3f}")

In [ ]:
# Visualize SOPR distribution
fig = make_subplots(rows=1, cols=2, subplot_titles=['SOPR Distribution', 'STH SOPR Distribution'])

fig.add_trace(go.Histogram(x=df['sopr'], nbinsx=100, name='SOPR'), row=1, col=1)
fig.add_vline(x=1, line_dash='dash', line_color='red', row=1, col=1)
fig.add_vline(x=1.02, line_dash='dot', line_color='green', row=1, col=1)
fig.add_vline(x=1.05, line_dash='dot', line_color='orange', row=1, col=1)

fig.add_trace(go.Histogram(x=df['sopr_sth'], nbinsx=100, name='STH SOPR'), row=1, col=2)
fig.add_vline(x=1, line_dash='dash', line_color='red', row=1, col=2)
fig.add_vline(x=1.02, line_dash='dot', line_color='green', row=1, col=2)
fig.add_vline(x=1.05, line_dash='dot', line_color='orange', row=1, col=2)

fig.update_layout(height=400, title_text='SOPR Distributions (Red=1, Green=1.02, Orange=1.05)')
fig.show()

In [ ]:
# Entry signal
both_below_1 = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
entries = both_below_1 & ~both_below_1.shift(1).fillna(False)

print(f"Entry signals: {entries.sum()}")

---
## Contrarian Exit Backtester

In [ ]:
def backtest_contrarian_exit(
    df: pd.DataFrame,
    entries: pd.Series,
    exit_sopr: float = 1.02,
    exit_sth_sopr: float = None,  # If None, only use SOPR
    exit_logic: str = 'or',  # 'or' = either triggers exit, 'and' = both required
    stop_loss: float = None,  # Optional price-based stop loss
    max_hold_days: int = 180,
):
    """Backtest with contrarian exit (sell when SOPR shows greed)."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_sopr = df['sopr'].iloc[j]
            current_sth_sopr = df['sopr_sth'].iloc[j]
            days_held = j - entry_idx
            
            # Check stop loss first
            if stop_loss is not None:
                pnl = (current_price - entry_price) / entry_price
                if pnl <= -stop_loss:
                    exit_date = current_date
                    exit_price = entry_price * (1 - stop_loss)
                    exit_reason = 'stop_loss'
                    break
            
            # Check contrarian exit
            sopr_exit = current_sopr >= exit_sopr
            sth_exit = current_sth_sopr >= exit_sth_sopr if exit_sth_sopr else False
            
            if exit_logic == 'or':
                exit_triggered = sopr_exit or sth_exit
            else:  # 'and'
                exit_triggered = sopr_exit and sth_exit
            
            if exit_triggered:
                exit_date = current_date
                exit_price = current_price
                if sopr_exit and sth_exit:
                    exit_reason = 'both_euphoria'
                elif sopr_exit:
                    exit_reason = 'sopr_euphoria'
                else:
                    exit_reason = 'sth_euphoria'
                break
            
            # Check max hold
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'entry_price': entry_price,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason,
            'entry_sopr': df['sopr'].iloc[entry_idx],
            'exit_sopr': df.loc[exit_date, 'sopr'] if exit_date in df.index else np.nan
        })
        
        # Skip entries during this trade
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
def calc_stats(trades):
    """Calculate strategy stats."""
    if len(trades) == 0:
        return {'n_trades': 0, 'total_return': 0, 'win_rate': 0, 'profit_factor': 0, 'avg_days': 0}
    
    total_return = (1 + trades['pnl_pct']).prod() - 1
    win_rate = (trades['pnl_pct'] > 0).mean()
    avg_win = trades[trades['pnl_pct'] > 0]['pnl_pct'].mean() if (trades['pnl_pct'] > 0).any() else 0
    avg_loss = trades[trades['pnl_pct'] <= 0]['pnl_pct'].mean() if (trades['pnl_pct'] <= 0).any() else 0
    
    gross_win = trades[trades['pnl_pct'] > 0]['pnl_pct'].sum()
    gross_loss = abs(trades[trades['pnl_pct'] <= 0]['pnl_pct'].sum())
    profit_factor = gross_win / gross_loss if gross_loss > 0 else np.inf
    
    return {
        'n_trades': len(trades),
        'total_return': total_return,
        'win_rate': win_rate,
        'avg_win': avg_win,
        'avg_loss': avg_loss,
        'profit_factor': profit_factor,
        'avg_days': trades['days_held'].mean()
    }

---
## Test Different Exit Thresholds

In [ ]:
# Test various SOPR exit thresholds
exit_thresholds = [
    {'name': 'SOPR > 1.00', 'sopr': 1.00, 'sth': None},
    {'name': 'SOPR > 1.01', 'sopr': 1.01, 'sth': None},
    {'name': 'SOPR > 1.02', 'sopr': 1.02, 'sth': None},
    {'name': 'SOPR > 1.03', 'sopr': 1.03, 'sth': None},
    {'name': 'SOPR > 1.05', 'sopr': 1.05, 'sth': None},
    {'name': 'STH > 1.00', 'sopr': 99, 'sth': 1.00},  # 99 = effectively disabled
    {'name': 'STH > 1.02', 'sopr': 99, 'sth': 1.02},
    {'name': 'STH > 1.05', 'sopr': 99, 'sth': 1.05},
    {'name': 'SOPR>1.02 OR STH>1.02', 'sopr': 1.02, 'sth': 1.02},
    {'name': 'SOPR>1.02 OR STH>1.05', 'sopr': 1.02, 'sth': 1.05},
    {'name': 'SOPR>1.05 OR STH>1.05', 'sopr': 1.05, 'sth': 1.05},
]

results = []

for config in exit_thresholds:
    trades = backtest_contrarian_exit(
        df=df,
        entries=entries,
        exit_sopr=config['sopr'],
        exit_sth_sopr=config['sth'],
        exit_logic='or',
        stop_loss=None,  # Pure contrarian - no price stops
        max_hold_days=180
    )
    
    stats = calc_stats(trades)
    stats['name'] = config['name']
    
    # Exit reason breakdown
    if len(trades) > 0:
        stats['euphoria_exits'] = (trades['exit_reason'].str.contains('euphoria')).sum()
        stats['max_hold_exits'] = (trades['exit_reason'] == 'max_hold').sum()
    else:
        stats['euphoria_exits'] = 0
        stats['max_hold_exits'] = 0
    
    results.append(stats)

results_df = pd.DataFrame(results)

print("CONTRARIAN EXIT COMPARISON (No Stop Loss)")
print("="*110)
print(f"{'Exit Rule':<25} {'Trades':>7} {'Return':>10} {'Win%':>7} {'AvgWin':>8} {'AvgLoss':>8} {'PF':>6} {'Days':>6} {'Euph':>6} {'MaxH':>6}")
print("-"*110)
for _, row in results_df.iterrows():
    print(f"{row['name']:<25} {row['n_trades']:>7} {row['total_return']*100:>9.0f}% {row['win_rate']*100:>6.0f}% "
          f"{row['avg_win']*100:>7.1f}% {row['avg_loss']*100:>7.1f}% {row['profit_factor']:>6.2f} "
          f"{row['avg_days']:>6.0f} {row['euphoria_exits']:>6} {row['max_hold_exits']:>6}")

In [ ]:
# Now test WITH a stop loss backup
print("\n\nCONTRARIAN EXIT + 15% STOP LOSS BACKUP")
print("="*110)

results_with_sl = []

for config in exit_thresholds:
    trades = backtest_contrarian_exit(
        df=df,
        entries=entries,
        exit_sopr=config['sopr'],
        exit_sth_sopr=config['sth'],
        exit_logic='or',
        stop_loss=0.15,  # 15% stop loss as backup
        max_hold_days=180
    )
    
    stats = calc_stats(trades)
    stats['name'] = config['name']
    
    if len(trades) > 0:
        stats['euphoria_exits'] = (trades['exit_reason'].str.contains('euphoria')).sum()
        stats['stop_exits'] = (trades['exit_reason'] == 'stop_loss').sum()
        stats['max_hold_exits'] = (trades['exit_reason'] == 'max_hold').sum()
    else:
        stats['euphoria_exits'] = 0
        stats['stop_exits'] = 0
        stats['max_hold_exits'] = 0
    
    results_with_sl.append(stats)

results_sl_df = pd.DataFrame(results_with_sl)

print(f"{'Exit Rule':<25} {'Trades':>7} {'Return':>10} {'Win%':>7} {'PF':>6} {'Days':>6} {'Euph':>6} {'Stop':>6} {'MaxH':>6}")
print("-"*110)
for _, row in results_sl_df.iterrows():
    print(f"{row['name']:<25} {row['n_trades']:>7} {row['total_return']*100:>9.0f}% {row['win_rate']*100:>6.0f}% "
          f"{row['profit_factor']:>6.2f} {row['avg_days']:>6.0f} "
          f"{row['euphoria_exits']:>6} {row['stop_exits']:>6} {row['max_hold_exits']:>6}")

---
## Walk-Forward Validation

In [ ]:
def walk_forward_contrarian(df, entries, exit_sopr, exit_sth_sopr, stop_loss, train_days=365, test_days=90, step_days=90):
    """Walk-forward for contrarian exit."""
    wf_results = []
    close = df['price']
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        if test_end <= test_start:
            break
        
        test_df = df.iloc[test_start:test_end]
        test_entries = entries.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        trades = backtest_contrarian_exit(
            df=test_df,
            entries=test_entries,
            exit_sopr=exit_sopr,
            exit_sth_sopr=exit_sth_sopr,
            exit_logic='or',
            stop_loss=stop_loss,
            max_hold_days=180
        )
        
        if len(trades) > 0:
            strat_return = (1 + trades['pnl_pct']).prod() - 1
            n_trades = len(trades)
        else:
            strat_return = 0
            n_trades = 0
        
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        wf_results.append({
            'fold': fold,
            'period': df.index[test_start].strftime('%Y-%m'),
            'n_trades': n_trades,
            'strat_return': strat_return,
            'hold_return': hold_return,
            'excess': strat_return - hold_return,
            'beat_hold': strat_return > hold_return
        })
    
    return pd.DataFrame(wf_results)

In [ ]:
# Walk-forward test top strategies
strategies_to_test = [
    {'name': 'Baseline (Trailing Stop)', 'type': 'trailing'},
    {'name': 'SOPR > 1.02 (pure)', 'sopr': 1.02, 'sth': None, 'sl': None},
    {'name': 'SOPR > 1.02 + 15% SL', 'sopr': 1.02, 'sth': None, 'sl': 0.15},
    {'name': 'SOPR > 1.03 + 15% SL', 'sopr': 1.03, 'sth': None, 'sl': 0.15},
    {'name': 'SOPR > 1.05 + 15% SL', 'sopr': 1.05, 'sth': None, 'sl': 0.15},
    {'name': 'STH > 1.02 + 15% SL', 'sopr': 99, 'sth': 1.02, 'sl': 0.15},
    {'name': 'SOPR>1.02 OR STH>1.02 + SL', 'sopr': 1.02, 'sth': 1.02, 'sl': 0.15},
]

wf_comparison = []

# Baseline trailing stop
def backtest_trailing_stop(close, entries, stop_loss=0.08, trailing_stop=0.12, min_profit=0.05, max_hold=180):
    trades = []
    entry_indices = entries[entries].index.tolist()
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = close.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        current_stop = entry_price * (1 - stop_loss)
        is_trailing = False
        
        exit_date = None
        exit_price = None
        
        for j in range(entry_idx + 1, len(close)):
            current_date = close.index[j]
            current_price = close.iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            
            if not is_trailing and pnl >= min_profit:
                is_trailing = True
            
            if is_trailing:
                trail_stop = peak_price * (1 - trailing_stop)
                if trail_stop > current_stop:
                    current_stop = trail_stop
            
            if current_price <= current_stop:
                exit_date = current_date
                exit_price = current_stop
                break
            
            if days_held >= max_hold:
                exit_date = current_date
                exit_price = current_price
                break
        
        if exit_date is None:
            exit_date = close.index[-1]
            exit_price = close.iloc[-1]
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({'entry_date': entry_date, 'exit_date': exit_date, 'pnl_pct': pnl})
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

# Baseline walk-forward
wf_baseline = []
total_days = len(df)
train_days = 365
test_days = 90
step_days = 90

for fold in range((total_days - train_days) // step_days):
    test_start = train_days + fold * step_days
    test_end = min(test_start + test_days, total_days)
    
    test_close = close.iloc[test_start:test_end]
    test_entries = entries.iloc[test_start:test_end]
    
    trades = backtest_trailing_stop(test_close, test_entries)
    strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
    hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
    
    wf_baseline.append({
        'strat_return': strat_return,
        'hold_return': hold_return,
        'beat_hold': strat_return > hold_return
    })

wf_baseline_df = pd.DataFrame(wf_baseline)
wf_comparison.append({
    'strategy': 'Baseline (Trailing Stop)',
    'beat_rate': wf_baseline_df['beat_hold'].mean(),
    'avg_excess': (pd.Series([r['strat_return'] for r in wf_baseline]) - pd.Series([r['hold_return'] for r in wf_baseline])).mean()
})

# Contrarian strategies
for strat in strategies_to_test[1:]:  # Skip baseline
    wf = walk_forward_contrarian(
        df, entries,
        exit_sopr=strat['sopr'],
        exit_sth_sopr=strat['sth'],
        stop_loss=strat['sl']
    )
    
    wf_comparison.append({
        'strategy': strat['name'],
        'beat_rate': wf['beat_hold'].mean(),
        'avg_excess': wf['excess'].mean()
    })

wf_comp_df = pd.DataFrame(wf_comparison).sort_values('beat_rate', ascending=False)

print("\nWALK-FORWARD COMPARISON")
print("="*70)
print(f"{'Strategy':<35} {'Beat Rate':>15} {'Avg Excess':>15}")
print("-"*70)
for _, row in wf_comp_df.iterrows():
    print(f"{row['strategy']:<35} {row['beat_rate']*100:>14.0f}% {row['avg_excess']*100:>+14.1f}%")

In [ ]:
# Visualize comparison
fig = go.Figure()

fig.add_trace(go.Bar(
    x=wf_comp_df['strategy'],
    y=wf_comp_df['beat_rate'] * 100,
    marker_color=['green' if x > 0.55 else 'orange' if x > 0.50 else 'red' for x in wf_comp_df['beat_rate']],
    text=[f"{x:.0f}%" for x in wf_comp_df['beat_rate']*100],
    textposition='outside'
))

fig.add_hline(y=50, line_dash='dash', line_color='red', annotation_text='50% (coin flip)')
fig.add_hline(y=54, line_dash='dot', line_color='orange', annotation_text='54% (baseline)')

fig.update_layout(
    title='Walk-Forward Beat Rate by Exit Strategy',
    yaxis_title='Beat Buy & Hold %',
    xaxis_tickangle=-45,
    height=500
)
fig.show()

---
## Look at Individual Trades

In [ ]:
# Get trades for best contrarian strategy
best_trades = backtest_contrarian_exit(
    df=df,
    entries=entries,
    exit_sopr=1.02,
    exit_sth_sopr=None,
    exit_logic='or',
    stop_loss=0.15,
    max_hold_days=180
)

print("\nTRADE DETAILS: SOPR > 1.02 + 15% SL")
print("="*120)

display_trades = best_trades.copy()
display_trades['entry_date'] = pd.to_datetime(display_trades['entry_date']).dt.strftime('%Y-%m-%d')
display_trades['exit_date'] = pd.to_datetime(display_trades['exit_date']).dt.strftime('%Y-%m-%d')
display_trades['entry_price'] = display_trades['entry_price'].round(0).astype(int)
display_trades['exit_price'] = display_trades['exit_price'].round(0).astype(int)
display_trades['pnl_pct'] = (display_trades['pnl_pct'] * 100).round(1)
display_trades['entry_sopr'] = display_trades['entry_sopr'].round(3)
display_trades['exit_sopr'] = display_trades['exit_sopr'].round(3)

print(display_trades.to_string(index=False))

In [ ]:
# Exit reason breakdown
print("\nEXIT REASON BREAKDOWN")
print("="*50)
print(best_trades['exit_reason'].value_counts())

# Win rate by exit reason
print("\nWIN RATE BY EXIT REASON")
print("-"*50)
for reason in best_trades['exit_reason'].unique():
    subset = best_trades[best_trades['exit_reason'] == reason]
    win_rate = (subset['pnl_pct'] > 0).mean()
    avg_pnl = subset['pnl_pct'].mean()
    print(f"{reason}: {len(subset)} trades, {win_rate*100:.0f}% win rate, {avg_pnl*100:.1f}% avg PnL")

---
## Summary

In [ ]:
print("\n" + "="*70)
print("CONTRARIAN EXIT STRATEGY SUMMARY")
print("="*70)

# Find best strategy
best_strat = wf_comp_df.iloc[0]

print(f"\n🏆 BEST STRATEGY: {best_strat['strategy']}")
print(f"   Walk-Forward Beat Rate: {best_strat['beat_rate']*100:.0f}%")
print(f"   Avg Excess Return: {best_strat['avg_excess']*100:+.1f}%")

print(f"\n📊 COMPARISON")
baseline_rate = wf_comp_df[wf_comp_df['strategy'] == 'Baseline (Trailing Stop)']['beat_rate'].values[0]
print(f"   Baseline (Trailing Stop): {baseline_rate*100:.0f}%")
print(f"   Best Contrarian: {best_strat['beat_rate']*100:.0f}%")
print(f"   Improvement: {(best_strat['beat_rate'] - baseline_rate)*100:+.0f}%")

print("\n" + "="*70)

In [ ]:
# Save results
import json

contrarian_results = {
    'strategy': 'contrarian_exit',
    'entry': 'SOPR < 1 AND STH_SOPR < 1',
    'exit': 'SOPR > threshold OR stop_loss',
    'walk_forward_comparison': wf_comp_df.to_dict('records'),
    'best_strategy': {
        'name': best_strat['strategy'],
        'beat_rate': float(best_strat['beat_rate']),
        'avg_excess': float(best_strat['avg_excess'])
    }
}

with open('../data/sopr_contrarian_exit_results.json', 'w') as f:
    json.dump(contrarian_results, f, indent=2, default=str)

print("Saved to ../data/sopr_contrarian_exit_results.json")